# 01 — Reprodução do detector de mixers na Ethereum

**Objetivo:** reconstruir de forma didática a execução experimental baseada no artigo *Uma Proposta de Algoritmo para a Detecção de Mixers na Ethereum* (Leale & Sendin, SBSeg 2025), **sem substituir os arquivos originais**.

**O que este notebook executa agora:** lê os datasets já preparados no Drive, verifica hashes, descreve a amostragem, reproduz a configuração Random Forest do projeto e avalia as janelas de 29/10/2020. **Não baixa o dump bruto nem refaz a inspeção manual dos rótulos** — essas etapas serão consolidadas e testadas à parte.

**Separação científica importante:** as características e decisões de processamento são escolhas de nossa reprodução; não devem ser atribuídas ao artigo quando não estão descritas nele.

**Código:** `src/ethmix/` no [GitHub](https://github.com/BorgesGusta/ethereum-mixer-detection). **Dados grandes:** [Drive próprio — Projeto_Mixers_Ethereum](https://drive.google.com/drive/folders/1Dkz-82gKd7q3woGZDjEHl9IhQNnC7hgj). A antiga pasta `mixers-ethereum` do Bruno é fonte histórica, não base ativa. **Histórico:** `notebooks/legacy/`.

> Os conjuntos de 2020 e 2026 já foram inspecionados em experimentos anteriores. Este notebook é reprodução/exploração, **não avaliação cega**.

## 1. A pergunta e a unidade de análise

A tarefa é **detectar janelas de atividade que se parecem com operações de mixers**, utilizando quatro características. A unidade do nosso experimento é `(recipient, dia UTC)`: um contrato pode produzir várias janelas em dias distintos.

O classificador não recebe transações isoladas como amostras. Ele recebe o **resumo estatístico de várias transações** por contrato/dia.

| Característica | Explicação |
|---|---|
| `value_entropy` | Diversidade dos valores transferidos |
| `fee_entropy` | Diversidade das taxas |
| `internal_value_entropy` | Diversidade dos valores internos da execução |
| `call_count_mean` | Média de chamadas por transação |

As entropias usam Shannon base 2, após arredondamento de valores (4 decimais para valor e valor interno; 6 para taxa).

## 2. Preparar o ambiente e carregar módulos

**GitHub privado:** use uma das opções:
- **A.** Se você já clonou o repositório no ambiente, defina `REPO_LOCAL` para essa pasta.
- **B.** No Colab, cadastre um token GitHub **somente leitura para este repositório**, nos *Secrets* (ícone de chave à esquerda), com o nome `GITHUB_TOKEN`. Dê permissão de acesso ao notebook. **Nunca cole token numa célula.**

O carregador abaixo baixa somente três arquivos Python para o diretório temporário do Colab. O token não é salvo em Drive nem incluído nos resultados.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, importlib, urllib.request, base64

REPO_LOCAL = None  # exemplo: '/content/ethereum-mixer-detection', se já estiver clonado
REPO = "BorgesGusta/ethereum-mixer-detection"
REVISAO = "main"  # para reprodução final, substituir por SHA do commit aprovado

if REPO_LOCAL:
    root = Path(REPO_LOCAL)
    if not (root / "src/ethmix/features.py").exists():
        raise FileNotFoundError("REPO_LOCAL não contém src/ethmix/features.py")
else:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    if not token:
        raise RuntimeError(
            "Repositório privado: configure GITHUB_TOKEN em Secrets do Colab, "
            "ou informe REPO_LOCAL com uma cópia do GitHub já clonada."
        )
    root = Path("/content/ethmix_readonly")
    for rel in ["src/ethmix/__init__.py", "src/ethmix/features.py", "src/ethmix/models.py"]:
        url = f"https://api.github.com/repos/{REPO}/contents/{rel}?ref={REVISAO}"
        req = urllib.request.Request(url, headers={
            "Authorization": "Bearer " + token,
            "Accept": "application/vnd.github+json",
            "X-GitHub-Api-Version": "2022-11-28"})
        with urllib.request.urlopen(req, timeout=30) as response:
            payload = json.load(response)
        local = root / rel
        local.parent.mkdir(parents=True, exist_ok=True)
        local.write_bytes(base64.b64decode(payload["content"]))
    del token
sys.path.insert(0, str(root / "src"))
import ethmix
from ethmix.features import FEATURES, DECIMAIS, calcular_features, preparar
from ethmix.models import novo_rf, validar_cv, reamostrar
print("Módulos de pesquisa carregados:", list(FEATURES))
print("Referência de código:", REVISAO)

## 3. Localizar as fontes no Drive com segurança

O Colab pode conter **várias cópias** do projeto. Se isso ocorrer, selecione `ESCOLHA_INDICE` após inspecionar os caminhos impressos. O notebook compara hashes dos datasets conhecidos e interrompe se forem diferentes.

Nenhum arquivo `.parquet` será escrito ou sobrescrito.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import pandas as pd, numpy as np
from datetime import datetime, timezone

# Pasta oficial do projeto (NÃO usar o atalho compartilhado do Bruno).
BASE_MANUAL = None  # se o Drive estiver montado em outro local, informe o caminho explicitamente
BASE_OFICIAL = Path("/content/drive/MyDrive/Segurança de Redes de Computadores/Projeto_Mixers_Ethereum")
BASE = Path(BASE_MANUAL) if BASE_MANUAL else BASE_OFICIAL
if not BASE.is_dir():
    raise FileNotFoundError(
        f"Pasta oficial não encontrada: {BASE}. Monte o Drive correto "
        "ou defina BASE_MANUAL. Não usar a pasta mixers-ethereum do Bruno.")
for arquivo in ["data/processed/features_treino.parquet",
                "data/processed/features_teste_20201029.parquet"]:
    if not (BASE / arquivo).is_file():
        raise FileNotFoundError(f"Arquivo ausente na pasta oficial: {BASE / arquivo}")
print("Pasta própria selecionada:", BASE)
INPUTS={
    "treino":BASE/"data/processed/features_treino.parquet",
    "teste2020":BASE/"data/processed/features_teste_20201029.parquet",
}
HASHES_ESPERADOS={
    "treino":"0a8baa11cee97d338d7a8d92b984f38192ba1a10ec915cc4a06d67ec94ad5c32",
    "teste2020":"6f8fd4e83e816c785d449116699065ad77425c9da6e7b19d9bcda36f42bddb51",
}
def sha256(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for bloco in iter(lambda:f.read(4*1024*1024),b""):
            h.update(bloco)
    return h.hexdigest()
for nome, path in INPUTS.items():
    atual=sha256(path)
    print(nome,path.name,atual)
    if atual!=HASHES_ESPERADOS[nome]:
        raise ValueError(f"SHA-256 divergente para {nome}. Não continuar sem auditar.")
print("Integridade: fontes verificadas.")

## 4. Compreender os dados de treinamento

O treino legado **não usa exclusivamente dados de 2020**. Contém 123 janelas: 23 positivas de março/2025 (do mesmo pool Tornado Cash 10 ETH) e 100 negativas selecionadas por inspeção de endereços.

Por isso, uma validação aleatória de linhas pode ser otimista: exemplos do **mesmo contrato** aparecem em treino e validação.

O filtro `MIN_TX=10` estabelece elegibilidade para inferência; ele não transforma linhas sem rótulo em negativas.

In [ ]:
treino=pd.read_parquet(INPUTS["treino"])
teste=pd.read_parquet(INPUTS["teste2020"])
MIN_TX=10
SEED=42

assert set(FEATURES).issubset(treino.columns)
assert set(FEATURES).issubset(teste.columns)
assert {"label","n_tx"}.issubset(treino.columns)
assert {"recipient","n_tx","tornado"}.issubset(teste.columns)
assert set(treino["label"].dropna().unique()) == {0,1}

dados_train=treino.loc[treino.n_tx>=MIN_TX].copy()
dados_2020=teste.loc[teste.n_tx>=MIN_TX].copy()
print("Treino completo:",treino.shape,"classes:",treino.label.value_counts().to_dict())
print("Treino elegível:",len(dados_train),"classes:",dados_train.label.value_counts().to_dict())
print("Janelas de 2020 antes de filtro:",len(teste))
print("Janelas elegíveis de 2020:",len(dados_2020))
print("Pools conhecidos com rótulo de referência no arquivo de teste:",
      int(dados_2020.tornado.notna().sum()))
display(dados_train.groupby("label")[list(FEATURES)].median().round(3))

## 5. Validar e treinar a Random Forest

Configuração usada na reprodução: **100 árvores**, profundidade máxima **5**, `class_weight='balanced'`, semente 42. Para comparar com o legado, realizamos uma validação cruzada estratificada de cinco folds com reamostragem **somente no treino de cada fold**.

**Atenção:** F1 alto neste conjunto não garante detecção de novos contratos nem estima a precisão de todos os endereços na blockchain.

In [ ]:
cv=validar_cv(dados_train,seed=SEED,n_splits=5)
print("Validação cruzada estratificada (exploratória):")
display(cv.round(3))
display(cv[["precision","recall","f1"]].agg(["mean","std"]).round(3))
Xt,yt=reamostrar(dados_train[list(FEATURES)],dados_train["label"].astype(int),SEED)
modelo=novo_rf(SEED).fit(Xt,yt)
print("Parâmetros RF:",modelo.get_params()["n_estimators"],
      modelo.get_params()["max_depth"])

## 6. Aplicar a 29/10/2020 sem pressupor rótulos negativos

Testamos dois limiares, **0,50 e 0,85**, apenas para descrição exploratória. A seleção de limiar para um futuro teste confirmatório deverá ser feita em uma validação independente previamente definida.

A coluna `tornado` marca somente pools conhecidos. Os demais endereços são **não rotulados**, não falsos positivos verificados.

In [ ]:
p=modelo.predict_proba(dados_2020[list(FEATURES)])[:,1]
resultados=[]
for limiar in [0.50,0.85]:
    flag=p>limiar
    res={
        "limiar":limiar,
        "enderecos_sinalizados":int(flag.sum()),
        "pools_referencia_recuperados":int(dados_2020.loc[flag,"tornado"].notna().sum()),
        "outros_sinalizados_sem_rotulo_confirmado":
            int(flag.sum()-dados_2020.loc[flag,"tornado"].notna().sum())
    }
    resultados.append(res)
resumo=pd.DataFrame(resultados)
display(resumo)
print("Os resultados dependem do treino reconstruído; confrontar com CSVs legados, não presumir igualdade.")
print("Número do artigo (63) é uma referência de comparação, não uma meta de otimização.")

## 7. Interpretar o resultado e registrar a execução

**Perguntas de defesa oral:** por que a unidade é contrato-dia? Por que o treino com positivos de um só pool limita a generalização? Qual é a diferença entre candidato adicional e falso positivo comprovado? Por que o threshold precisa ser calibrado sem olhar o teste?

Na primeira execução desta versão, registremos hashes e métricas para comparar com os resultados legados. **Não gravamos modelos nem datasets sobre os anteriores.**

In [ ]:
SALVAR_RELATORIO=True
if SALVAR_RELATORIO:
    ts=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    OUT=BASE/"results"/"reproducao_consolidada"/ts
    OUT.mkdir(parents=True,exist_ok=False)
    cv.to_csv(OUT/"cv_folds.csv",index=False)
    resumo.to_csv(OUT/"resumo_2020.csv",index=False)
    manifesto={
        "experimento":"reproducao_consolidada_01",
        "data_utc":ts,
        "repo":REPO,
        "codigo_ref":REVISAO,
        "dados_sha256":HASHES_ESPERADOS,
        "min_tx":MIN_TX,"seed":SEED,
        "rf":{"n_estimators":100,"max_depth":5,"class_weight":"balanced"},
        "limiares_exploratorios":[0.50,0.85],
        "ressalvas":["positivos concentrados em um contrato",
                     "enderecos nao rotulados nao sao negativos",
                     "teste 2020 ja foi explorado anteriormente"]
    }
    (OUT/"manifesto.json").write_text(json.dumps(manifesto,indent=2,ensure_ascii=False),encoding="utf-8")
    print("Relatórios novos:",OUT)
print("Fim da etapa didática. Não foi executada reconstrução integral desde dumps.")